<a href="https://colab.research.google.com/github/sergioGarcia91/SeismicUP/blob/main/06_MecanismosFocales_ISC_CMT_SG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://github.com/andrebelem/pythonverse/blob/main/PyGMT_0_17_0_GoogleColab_Setup.ipynb

En este notebook se realizará la representación gráfica de los mecanismos focales descargados de las plataformas ISC y Global CMT:

- ISC: https://www.isc.ac.uk/iscbulletin/search/fmechanisms/

- Global CMT: https://www.globalcmt.org/


| Catálogo | Longitud (min, max) | Latitud (min, max) |
| :------: | :-----------------: | :----------------: |
|    CC    |    -75.38, -72.41   |     5.72, 8.14     |


# Inicio

In [ ]:
!git clone https://github.com/sergioGarcia91/SeismicUP.git

In [ ]:
!pip install obspy

In [ ]:
!pip3 install contextily

In [ ]:
!apt-get install -y unrar

In [ ]:
!pip install rarfile

In [ ]:
# para incluir la libreria clonada
import sys
sys.path.append("/content/SeismicUP")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import seismicup as sup
import contextily as cx #para el basemap en geopandas
import xyzservices.providers as xyz #para escoger el basemap
import glob
import geopandas as gpd
import rarfile

from obspy.core import read
from obspy.imaging.beachball import beachball
from obspy.imaging.beachball import beach

# Conectar al Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Cambiar Fuente

In [ ]:
sup.plots.get_TimesNewRoman_font()

# Paths

In [ ]:
path_save_figures = ''

In [ ]:
archivo_rar = '/content/SeismicUP/Layers/MGN2020_MPIO_POLITICO.part01.rar'
with rarfile.RarFile(archivo_rar) as rf:
    rf.extractall('/content/SeismicUP/Layers')

path_shape_municipios = '/content/SeismicUP/Layers'

In [ ]:
path_fallas = '/content/SeismicUP/Layers/Fallas_colombia.shp'

In [ ]:
path_aoi = '/content/SeismicUP/Layers/Areas_Manuscrito.shp'

In [ ]:
path_datasets = '/content/SeismicUP/Datasets_/Cat_mec_focal_tensor_mom_SGC_2014_2024'
os.listdir(path_datasets)

# Cargar catalogos

In [ ]:
cvs_mecanismos_focales = 'Cat_mec_foc_CC_ISC_CMT.csv'

df = pd.read_csv(os.path.join(path_datasets, cvs_mecanismos_focales),
                 sep=';',
                 decimal=',')

df.head()

In [ ]:
df.columns

In [ ]:
df_mf = df[['lon', 'lat', 'depth', 'str1', 'dip1', 'rake1', 'Catalogo']].copy()
#-75.38, -72.41	5.72, 8.14
filtro_long = (df_mf['lon'] >= -75.38) & (df_mf['lon'] <= -72.41)
filtro_lat = (df_mf['lat'] >= 5.72) & (df_mf['lat'] <= 8.14)
df_mf = df_mf[filtro_long & filtro_lat]
df_mf.reset_index(drop=True, inplace=True)

df_mf

In [ ]:
df_mf.columns

## Plot

In [ ]:
shape_municipios = gpd.read_file(os.path.join(path_shape_municipios, 'MGN_MPIO_POLITICO.shp'))
shape_municipios = shape_municipios.to_crs(epsg=4326)

shape_municipios.boundary.plot()

In [ ]:
fallas_colombia = gpd.read_file(path_fallas)
fallas_colombia.to_crs(epsg=4326, inplace=True)
fallas_colombia.plot()

In [ ]:
puntoCC = [-73.731324, 6.792017]
puntoCC = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy([puntoCC[0]], [puntoCC[1]]),
    crs='EPSG:4326'
)

puntoCC.plot()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7), ncols=1, nrows=1)

for x, y, strike, dip, rake, prof in zip(df_mf['lon'],
                                         df_mf['lat'],
                                         df_mf['str1'],
                                         df_mf['dip1'],
                                         df_mf['rake1'],
                                         df_mf['depth']):
  focalmechs = [strike, dip, rake] # strike, dip, and rake

  if prof <= 10:
    color_ = 'b'
  elif prof <= 20:
    color_ = 'g'
  elif prof <= 40:
    color_ = 'y'
  elif prof <= 60:
    color_ = 'r'

  b = beach(focalmechs,
            facecolor=color_,
            edgecolor='k',
            width=50,
            linewidth=0.5,
            xy=(x,y),
            zorder=30,
            axes = ax)
            #outfile=os.path.join(path_save_figures,f'{nombre}.svg'),
            #format='svg')

  ax.add_collection(b)

shape_municipios.boundary.plot(color='grey',
                               ax=ax,
                               lw=1,
                               zorder=5)

fallas_colombia.plot(color='k',
                     ax=ax,
                     lw=1,
                     zorder=10,
                     label='Faults')

puntoCC.plot(color='r',
             ax=ax,
             zorder=20,
             label='Campo Colorado')

ax.set_xlim(-75.38-0.2, -72.41+0.2)
ax.set_ylim(5.72-0.2, 8.14+0.2)
cx.add_basemap(ax=ax,
               crs='epsg:4326', # el sistema de coordenadas
               source=xyz.OpenTopoMap,
               reset_extent=True,
               zorder=0,
               #zoom=10,
               attribution_size=1) # Para incluir un mapa base

ax.set_aspect('equal')

# plt.grid(True,
#          ls='--',
#          color='gray',
#          alpha=0.8)

legend  = plt.legend(loc='upper center')
legend.set_zorder(100)

plt.savefig(os.path.join(path_save_figures,'isc_cmt.png'),
            dpi=300,
            bbox_inches='tight')

plt.show()


# ISC - CMT - SGC

In [ ]:
cvs_mecanismos_focales = 'Cat_mec_foc_CC_ISC_CMT_SGC.csv'
df = pd.read_csv(os.path.join(path_datasets, cvs_mecanismos_focales),
                 sep=';',
                 decimal=',')

df.head()

In [ ]:
df = df.iloc[::-1]
df.head()

In [ ]:
df.columns

In [ ]:
df_mf = df[['lon', 'lat', 'depth', 'str1', 'dip1', 'rake1', 'Catalogo']].copy()
#-75.38, -72.41	5.72, 8.14
filtro_long = (df_mf['lon'] >= -75.38) & (df_mf['lon'] <= -72.41)
filtro_lat = (df_mf['lat'] >= 5.72) & (df_mf['lat'] <= 8.14)
df_mf = df_mf[filtro_long & filtro_lat]
df_mf.reset_index(drop=True, inplace=True)

df_mf

## Plot

In [ ]:
aoi = gpd.read_file(path_aoi)

aoi.boundary.plot()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7), ncols=1, nrows=1)

for x, y, strike, dip, rake, prof in zip(df_mf['lon'],
                                         df_mf['lat'],
                                         df_mf['str1'],
                                         df_mf['dip1'],
                                         df_mf['rake1'],
                                         df_mf['depth']):
  focalmechs = [strike, dip, rake] # strike, dip, and rake

  if prof <= 10:
    color_ = 'b'
  elif prof <= 20:
    color_ = 'g'
  elif prof <= 40:
    color_ = 'y'
  elif prof <= 60:
    color_ = 'r'

  b = beach(focalmechs,
            facecolor=color_,
            edgecolor='k',
            width=50,
            linewidth=0.5,
            xy=(x,y),
            zorder=30,
            axes = ax)
            #outfile=os.path.join(path_save_figures,f'{nombre}.svg'),
            #format='svg')

  ax.add_collection(b)

fallas_colombia.plot(color='k',
                     ax=ax,
                     lw=1,
                     zorder=10,
                     label='Faults')

aoi.boundary.plot(color='b',
                  ax=ax,
                  lw=1.5,
                  ls='--',
                  zorder=15,)

puntoCC.plot(color='r',
             ax=ax,
             zorder=20,
             label='Campo Colorado')


ax.set_xlim(-75.38-0.2, -72.41+0.2)
ax.set_ylim(5.72-0.2, 8.14+0.2)
cx.add_basemap(ax=ax,
               crs='epsg:4326', # el sistema de coordenadas
               source=xyz.OpenTopoMap,
               reset_extent=True,
               zorder=0,
               #zoom=10,
               attribution_size=1) # Para incluir un mapa base

ax.set_aspect('equal')

# plt.grid(True,
#          ls='--',
#          color='gray',
#          alpha=0.8)

legend  = plt.legend(loc='upper center')
legend.set_zorder(100)

plt.savefig(os.path.join(path_save_figures,'isc_cmt_sgc.png'),
            dpi=300,
            bbox_inches='tight')

plt.show()


# ISC - CMT - SGC - SG

In [ ]:
#cvs_mecanismos_focales = 'Cat_mec_foc_CC_ISC_CMT_SGC_SG_shape.csv'
# df = pd.read_csv(os.path.join(path_datasets, cvs_mecanismos_focales),
#                  sep=';',
#                  decimal=',')
cvs_mecanismos_focales = '/content/SeismicUP/Outputs/MecFoc_CC_ISC_CMT_SGC_SG.csv'
df = pd.read_csv(cvs_mecanismos_focales,
                 sep=';',
                 decimal=',')

df.head()

In [ ]:
df.columns

In [ ]:
df_mf = df[['lon', 'lat', 'depth', 'str1', 'dip1', 'rake1', 'Catalogo']].copy()
#-75.38, -72.41	5.72, 8.14
filtro_long = (df_mf['lon'] >= -75.38) & (df_mf['lon'] <= -72.41)
filtro_lat = (df_mf['lat'] >= 5.72) & (df_mf['lat'] <= 8.14)
df_mf = df_mf[filtro_long & filtro_lat]
df_mf.reset_index(drop=True, inplace=True)

df_mf

In [ ]:
df_mf = df_mf[df_mf['depth'] <= 60]
df_mf.reset_index(drop=True, inplace=True)

df_mf

## Plot

In [ ]:
# Radio en metros
radio_m = 15000

# Crear círculo de 15 km alrededor de Campo Colorado
circulo_15km = puntoCC.to_crs(epsg=3116).buffer(radio_m)

circulo_15km = gpd.GeoDataFrame(
    geometry=circulo_15km,
    crs="EPSG:3116"
).to_crs(epsg=4326)

In [ ]:
circulo_15km.boundary.plot()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7), ncols=1, nrows=1)

for x, y, strike, dip, rake, prof in zip(df_mf['lon'],
                                         df_mf['lat'],
                                         df_mf['str1'],
                                         df_mf['dip1'],
                                         df_mf['rake1'],
                                         df_mf['depth']):
  focalmechs = [strike, dip, rake] # strike, dip, and rake

  if prof <= 10:
    color_ = 'b'
  elif prof <= 20:
    color_ = 'g'
  elif prof <= 40:
    color_ = 'y'
  elif prof <= 60:
    color_ = 'r'

  b = beach(focalmechs,
            facecolor=color_,
            edgecolor='k',
            width=80,
            linewidth=0.5,
            xy=(x,y),
            zorder=30,
            axes = ax)
            #outfile=os.path.join(path_save_figures,f'{nombre}.svg'),
            #format='svg')

  ax.add_collection(b)

fallas_colombia.plot(color='k',
                     ax=ax,
                     lw=1,
                     zorder=10,
                     label='Faults')

aoi.boundary.plot(color='b',
                  ax=ax,
                  lw=1.5,
                  ls='--',
                  zorder=15,)

puntoCC.plot(color='r',
             ax=ax,
             zorder=100,
             label='Campo Colorado')

circulo_15km.boundary.plot(
    ax=ax,
    color='red',
    lw=1.5,
    ls='--',
    zorder=100,
    label='Radio 15 km')

ax.set_xlim(-74.19-0.01, -73.26+0.01)
ax.set_ylim(6.32-0.01, 7.25+0.01)
cx.add_basemap(ax=ax,
               crs='epsg:4326', # el sistema de coordenadas
               source=xyz.OpenTopoMap,
               reset_extent=True,
               zorder=0,
               #zoom=10,
               attribution_size=1) # Para incluir un mapa base

ax.set_aspect('equal')

# plt.grid(True,
#          ls='--',
#          color='gray',
#          alpha=0.8)

legend  = plt.legend(loc='upper center')
legend.set_zorder(100)

plt.savefig(os.path.join(path_save_figures,'isc_cmt_sgc_sg.png'),
            dpi=300,
            bbox_inches='tight')

plt.show()


# Fin